In [16]:
import os, math, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_DIR = "/kaggle/input/datasets/vedantjadhavv/ett-data"
CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "batch_size": 64,
    "epochs": 50,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "early_stop_patience": 5,
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = CONFIG["device"]
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

cuda Tesla T4


In [17]:
for f in ["ETTh1", "ETTh2", "ETTm1", "ETTm2"]:
    df = pd.read_csv(f"{DATA_DIR}/{f}.csv")
    print(f, df.shape, df.columns.tolist())

ETTh1 (17420, 8) ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
ETTh2 (17420, 8) ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
ETTm1 (69680, 8) ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
ETTm2 (69680, 8) ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


In [18]:
class ETTSeriesDataset(Dataset):
    def __init__(self, csv_path, split="train", lookback=96, horizon=96,
                 train_ratio=0.7, val_ratio=0.1, norm_stats=None):
        df = pd.read_csv(csv_path)
        if df.columns[0].lower() in ["date", "time", "datetime"]:
            df = df.iloc[:, 1:]
        data = df.values.astype(np.float32)
        T, C = data.shape

        train_end = int(T * train_ratio)
        val_end = int(T * (train_ratio + val_ratio))
        if split == "train":
            data = data[:train_end]
        elif split == "val":
            data = data[train_end:val_end]
        elif split == "test":
            data = data[val_end:]
        else:
            raise ValueError("split must be train/val/test")

        if len(data) <= lookback + horizon:
            raise ValueError("not enough data for lookback/horizon")

        mean = norm_stats["mean"]; std = norm_stats["std"] + 1e-8
        data = (data - mean) / std

        self.data = data
        self.lookback = lookback
        self.horizon = horizon
        self.C = C

        self.X, self.Y = [], []
        for i in range(len(self.data) - lookback - horizon + 1):
            self.X.append(self.data[i:i+lookback])
            self.Y.append(self.data[i+lookback:i+lookback+horizon])
        self.X = np.stack(self.X)
        self.Y = np.stack(self.Y)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.Y[idx])


def get_ett_dataloaders(csv_path, lookback=96, horizon=96,
                         train_ratio=0.7, val_ratio=0.1,
                         batch_size=64, num_workers=2):
    df = pd.read_csv(csv_path)
    if df.columns[0].lower() in ["date", "time", "datetime"]:
        df = df.iloc[:, 1:]
    data = df.values.astype(np.float32)
    T = data.shape[0]
    train_end = int(T * train_ratio)
    train_data = data[:train_end]

    norm_stats = {"mean": train_data.mean(axis=0), "std": train_data.std(axis=0)}

    train_ds = ETTSeriesDataset(csv_path, "train", lookback, horizon, train_ratio, val_ratio, norm_stats)
    val_ds   = ETTSeriesDataset(csv_path, "val",   lookback, horizon, train_ratio, val_ratio, norm_stats)
    test_ds  = ETTSeriesDataset(csv_path, "test",  lookback, horizon, train_ratio, val_ratio, norm_stats)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader, norm_stats

In [10]:
class DLinear(nn.Module):
    def __init__(self, lookback, horizon, num_features, moving_avg_kernel=25):
        super().__init__()
        self.lookback = lookback; self.horizon = horizon
        self.moving_avg = nn.AvgPool1d(kernel_size=moving_avg_kernel, stride=1, padding=moving_avg_kernel//2)
        self.seasonal_linear = nn.Linear(lookback, horizon)
        self.trend_linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        B, L, C = x.shape
        x_bc = x.reshape(B*C, L).unsqueeze(1)
        trend = self.moving_avg(x_bc).squeeze(1)
        seasonal = x_bc.squeeze(1) - trend
        seasonal_out = self.seasonal_linear(seasonal)
        trend_out = self.trend_linear(trend)
        pred = (seasonal_out + trend_out).reshape(B, C, self.horizon)
        return pred.permute(0, 2, 1)


class PatchMLPTS(nn.Module):
    def __init__(self, lookback, horizon, num_features,
                 patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1):
        super().__init__()
        self.lookback = lookback; self.horizon = horizon
        self.patch_len = patch_len
        self.num_patches = (lookback - patch_len) // patch_len + 1

        self.patch_embed = nn.Linear(patch_len, embed_dim)
        self.mlp_blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim*4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim*4, embed_dim),
                nn.Dropout(dropout),
            ) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(embed_dim, horizon)

    def forward(self, x):
        B, L, C = x.shape
        P, N = self.patch_len, self.num_patches
        L_eff = N * P
        x = x[:, :L_eff, :]
        x = x.reshape(B, N, P, C)
        x = x.permute(0, 3, 1, 2).reshape(B*C, N, P)
        e = self.patch_embed(x)
        for block in self.mlp_blocks:
            e = e + block(e)
        z = e.mean(dim=1)
        y = self.head(z)
        return y.reshape(B, C, self.horizon).permute(0, 2, 1)

In [19]:
def compute_mse_mae(pred, target):
    mse = F.mse_loss(pred, target).item()
    mae = F.l1_loss(pred, target).item()
    return mse, mae

class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.sum = 0.0; self.count = 0
    def update(self, val, n=1): self.sum += val*n; self.count += n
    def avg(self): return self.sum / self.count if self.count > 0 else 0.0

In [20]:
class DLinear(nn.Module):
    def __init__(self, lookback, horizon, num_features, moving_avg_kernel=25):
        super().__init__()
        self.lookback = lookback; self.horizon = horizon
        self.moving_avg = nn.AvgPool1d(kernel_size=moving_avg_kernel, stride=1, padding=moving_avg_kernel//2)
        self.seasonal_linear = nn.Linear(lookback, horizon)
        self.trend_linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        B, L, C = x.shape
        x_bc = x.permute(0, 2, 1).reshape(B*C, L).unsqueeze(1)  # fixed: permute before reshape
        trend = self.moving_avg(x_bc).squeeze(1)
        seasonal = x_bc.squeeze(1) - trend
        seasonal_out = self.seasonal_linear(seasonal)
        trend_out = self.trend_linear(trend)
        pred = (seasonal_out + trend_out).reshape(B, C, self.horizon)
        return pred.permute(0, 2, 1)


class PatchMLPTS(nn.Module):
    def __init__(self, lookback, horizon, num_features,
                 patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1):
        super().__init__()
        self.lookback = lookback; self.horizon = horizon
        self.patch_len = patch_len
        self.num_patches = (lookback - patch_len) // patch_len + 1

        self.patch_embed = nn.Linear(patch_len, embed_dim)
        self.mlp_blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim*4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim*4, embed_dim),
                nn.Dropout(dropout),
            ) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(embed_dim, horizon)

    def forward(self, x):
        B, L, C = x.shape
        P, N = self.patch_len, self.num_patches
        L_eff = N * P
        x = x[:, :L_eff, :]
        x = x.reshape(B, N, P, C)
        x = x.permute(0, 3, 1, 2).reshape(B*C, N, P)
        e = self.patch_embed(x)
        for block in self.mlp_blocks:
            e = e + block(e)
        z = e.mean(dim=1)
        y = self.head(z)
        return y.reshape(B, C, self.horizon).permute(0, 2, 1)

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    mse_meter = AverageMeter()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            pred = model(x)
            loss = criterion(pred, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        mse, _ = compute_mse_mae(pred.float(), y)
        mse_meter.update(mse, x.size(0))
    return mse_meter.avg()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    mse_meter, mae_meter = AverageMeter(), AverageMeter()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        mse, mae = compute_mse_mae(pred, y)
        mse_meter.update(mse, x.size(0)); mae_meter.update(mae, x.size(0))
    return mse_meter.avg(), mae_meter.avg()

def fit_model(model, train_loader, val_loader, test_loader,
              epochs=50, lr=1e-3, weight_decay=1e-4, patience=5, device="cuda", verbose=True):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=patience//2)
    scaler = torch.amp.GradScaler('cuda')

    best_val_mse = float("inf"); best_state = None; patience_counter = 0

    for epoch in range(epochs):
        t0 = time.time()
        train_mse = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_mse, val_mae = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_mse)

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose:
            print(f"epoch {epoch+1:02d} | train {train_mse:.6f} | val {val_mse:.6f} | val_mae {val_mae:.6f} | {time.time()-t0:.1f}s")
        if patience_counter >= patience:
            if verbose: print("early stop")
            break

    model.load_state_dict(best_state)
    test_mse, test_mae = evaluate(model, test_loader, criterion, device)
    if verbose:
        print(f"TEST MSE: {test_mse:.6f} | TEST MAE: {test_mae:.6f}")
    return model, test_mse, test_mae

In [22]:
csv_path = f"{DATA_DIR}/ETTh1.csv"
lookback, horizon = 96, 96

train_loader, val_loader, test_loader, stats = get_ett_dataloaders(
    csv_path, lookback=lookback, horizon=horizon, batch_size=CONFIG["batch_size"]
)
num_features = train_loader.dataset.C
print("num_features:", num_features)

print("=== DLinear ===")
dlinear = DLinear(lookback, horizon, num_features, moving_avg_kernel=25)
dlinear, dl_mse, dl_mae = fit_model(dlinear, train_loader, val_loader, test_loader,
                                     epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                                     weight_decay=CONFIG["weight_decay"],
                                     patience=CONFIG["early_stop_patience"], device=device)

print("\n=== Patch-MLP-TS ===")
pmts = PatchMLPTS(lookback, horizon, num_features, patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1)
pmts, pm_mse, pm_mae = fit_model(pmts, train_loader, val_loader, test_loader,
                                  epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                                  weight_decay=CONFIG["weight_decay"],
                                  patience=CONFIG["early_stop_patience"], device=device)

print(f"\nDLinear: MSE={dl_mse:.4f} MAE={dl_mae:.4f}")
print(f"Patch-MLP-TS: MSE={pm_mse:.4f} MAE={pm_mae:.4f}")

num_features: 7
=== DLinear ===
epoch 01 | train 0.446849 | val 0.376675 | val_mae 0.415049 | 1.0s
epoch 02 | train 0.375096 | val 0.362022 | val_mae 0.398790 | 1.0s
epoch 03 | train 0.368743 | val 0.360160 | val_mae 0.397158 | 1.0s
epoch 04 | train 0.366137 | val 0.359780 | val_mae 0.398500 | 1.0s
epoch 05 | train 0.364828 | val 0.366990 | val_mae 0.407995 | 1.1s
epoch 06 | train 0.364130 | val 0.367567 | val_mae 0.407525 | 1.0s
epoch 07 | train 0.363630 | val 0.355023 | val_mae 0.391279 | 1.0s
epoch 08 | train 0.363285 | val 0.358660 | val_mae 0.395600 | 1.0s
epoch 09 | train 0.363070 | val 0.362579 | val_mae 0.403062 | 1.0s
epoch 10 | train 0.363197 | val 0.355848 | val_mae 0.394362 | 1.0s
epoch 11 | train 0.361652 | val 0.361708 | val_mae 0.400744 | 1.0s
epoch 12 | train 0.361586 | val 0.359201 | val_mae 0.399350 | 1.0s
early stop
TEST MSE: 0.441026 | TEST MAE: 0.445585

=== Patch-MLP-TS ===
epoch 01 | train 0.563096 | val 0.498920 | val_mae 0.489970 | 1.7s
epoch 02 | train 0.54733

In [25]:
datasets = ["ETTh1", "ETTh2", "ETTm1", "ETTm2"]
horizons = [96, 192, 336, 720]
lookback = 96  # keep fixed across all for fair Patch-MLP-TS vs DLinear comparison

all_results = []

for ds in datasets:
    csv_path = f"{DATA_DIR}/{ds}.csv"
    for h in horizons:
        print(f"\n{'='*50}\n{ds} | horizon={h}\n{'='*50}")
        train_loader, val_loader, test_loader, stats = get_ett_dataloaders(
            csv_path, lookback=lookback, horizon=h, batch_size=CONFIG["batch_size"]
        )
        num_features = train_loader.dataset.C

        for model_name, model_cls, kwargs in [
            ("DLinear", DLinear, {"moving_avg_kernel": 25}),
            ("PatchMLPTS", PatchMLPTS, {"patch_len": 12, "embed_dim": 128, "num_blocks": 3, "dropout": 0.1}),
        ]:
            model = model_cls(lookback, h, num_features, **kwargs)
            _, test_mse, test_mae = fit_model(
                model, train_loader, val_loader, test_loader,
                epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                patience=CONFIG["early_stop_patience"], device=device, verbose=False
            )
            print(f"{model_name}: MSE={test_mse:.4f} MAE={test_mae:.4f}")
            all_results.append({"dataset": ds, "horizon": h, "model": model_name,
                                 "MSE": test_mse, "MAE": test_mae})

df_results = pd.DataFrame(all_results)
df_results.to_csv("all_results.csv", index=False)
print("\nsaved to all_results.csv")
df_results


ETTh1 | horizon=96
DLinear: MSE=0.4382 MAE=0.4437
PatchMLPTS: MSE=0.8153 MAE=0.6477

ETTh1 | horizon=192
DLinear: MSE=0.4897 MAE=0.4815
PatchMLPTS: MSE=0.8572 MAE=0.6696

ETTh1 | horizon=336
DLinear: MSE=0.5326 MAE=0.5166
PatchMLPTS: MSE=0.8867 MAE=0.6856

ETTh1 | horizon=720
DLinear: MSE=0.6515 MAE=0.5927
PatchMLPTS: MSE=1.0099 MAE=0.7470

ETTh2 | horizon=96
DLinear: MSE=0.1685 MAE=0.2814
PatchMLPTS: MSE=0.2164 MAE=0.3247

ETTh2 | horizon=192
DLinear: MSE=0.1932 MAE=0.3076
PatchMLPTS: MSE=0.2794 MAE=0.3804

ETTh2 | horizon=336
DLinear: MSE=0.2114 MAE=0.3254
PatchMLPTS: MSE=0.2474 MAE=0.3550

ETTh2 | horizon=720
DLinear: MSE=0.2588 MAE=0.3694
PatchMLPTS: MSE=0.2766 MAE=0.3892

ETTm1 | horizon=96
DLinear: MSE=0.3545 MAE=0.3846
PatchMLPTS: MSE=0.8376 MAE=0.6265

ETTm1 | horizon=192
DLinear: MSE=0.4223 MAE=0.4215
PatchMLPTS: MSE=0.8559 MAE=0.6441

ETTm1 | horizon=336
DLinear: MSE=0.4916 MAE=0.4590
PatchMLPTS: MSE=0.8866 MAE=0.6539

ETTm1 | horizon=720
DLinear: MSE=0.5537 MAE=0.5022
Patch

,dataset,horizon,model,MSE,MAE
0,ETTh1,96,DLinear,0.438204,0.443728
1,ETTh1,96,PatchMLPTS,0.815329,0.647680
2,ETTh1,192,DLinear,0.489725,0.481548
3,ETTh1,192,PatchMLPTS,0.857216,0.669595
4,ETTh1,336,DLinear,0.532552,0.516558
5,ETTh1,336,PatchMLPTS,0.886677,0.685596
6,ETTh1,720,DLinear,0.651548,0.592721
7,ETTh1,720,PatchMLPTS,1.009853,0.747015
8,ETTh2,96,DLinear,0.168492,0.281425
9,ETTh2,96,PatchMLPTS,0.216398,0.324688


In [24]:
pmts = PatchMLPTS(lookback, horizon, num_features, patch_len=12, embed_dim=64, num_blocks=2, dropout=0.2)
pmts, pm_mse, pm_mae = fit_model(pmts, train_loader, val_loader, test_loader,
                                  epochs=CONFIG["epochs"], lr=5e-4,
                                  weight_decay=CONFIG["weight_decay"],
                                  patience=CONFIG["early_stop_patience"], device=device)
print(f"Patch-MLP-TS (smaller, lower lr): MSE={pm_mse:.4f} MAE={pm_mae:.4f}")

epoch 01 | train 0.595340 | val 0.498700 | val_mae 0.487698 | 1.5s
epoch 02 | train 0.551416 | val 0.497590 | val_mae 0.484983 | 1.5s
epoch 03 | train 0.548157 | val 0.498237 | val_mae 0.490971 | 1.5s
epoch 04 | train 0.546578 | val 0.498597 | val_mae 0.492763 | 1.6s
epoch 05 | train 0.544875 | val 0.497192 | val_mae 0.489678 | 1.5s
epoch 06 | train 0.544169 | val 0.496069 | val_mae 0.488680 | 1.5s
epoch 07 | train 0.543059 | val 0.504191 | val_mae 0.499027 | 1.5s
epoch 08 | train 0.542260 | val 0.498441 | val_mae 0.489539 | 1.4s
epoch 09 | train 0.541870 | val 0.498773 | val_mae 0.493029 | 1.5s
epoch 10 | train 0.539923 | val 0.499953 | val_mae 0.493821 | 1.4s
epoch 11 | train 0.539708 | val 0.500514 | val_mae 0.493533 | 1.5s
early stop
TEST MSE: 0.814065 | TEST MAE: 0.648367
Patch-MLP-TS (smaller, lower lr): MSE=0.8141 MAE=0.6484
